**V1 experiment.** This notebook contains the V1 implementation and experiment configuration. See [methods and evaluation](../docs/METHODS.md) for the implemented protocol.


# Level 2 — Delphinidae Species Classification (semi-supervised)

**Goal:** Test graph SSL on a harder problem (acoustically similar species),
and analyze sensitivity to graph construction parameters.

Pipeline:
1. Load `Level2/Data/level2_features.npz`, standardize features.
2. Use the **winning** method from Level 1 (read `Level1/winner.txt`).
3. 50% labels, 20 trials at default (k=10, σ=median).
4. Parameter analysis over k ∈ {3, 5, 7, 10, 15, 20} and
   σ ∈ {0.5×median, 1×median, 2×median, 5×median, self_tuning}.
5. Spectral gap at the best (k, σ).
6. Diffusion map (primary) + t-SNE (comparison) at default params.
7. ResNet-18 baseline.
8. Confusion matrix at best params.


In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, f1_score
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.graph import (
    build_knn_graph,
    compute_laplacian,
    spectral_gap_analysis,
    spectral_gap_from_W,
    diffusion_map_embedding,
    label_propagation,
    laplacian_regularization,
)
from utils.evaluation import (
    plot_confusion_matrix,
    plot_spectral_gap,
    plot_parameter_heatmap,
)
from utils.cnn import train_resnet_baseline

LEVEL_DIR = PROJECT_ROOT / "Level2"
DATA_DIR = LEVEL_DIR / "Data"
FIG_DIR = LEVEL_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

WINNER_FILE = PROJECT_ROOT / "Level1" / "winner.txt"
WINNER = WINNER_FILE.read_text(encoding="utf-8").strip() if WINNER_FILE.exists() else "label_prop"
print(f"Level 1 winner = {WINNER}")
graph_method = label_propagation if WINNER == "label_prop" else laplacian_regularization


## Load features


In [ ]:
data = np.load(DATA_DIR / "level2_features.npz", allow_pickle=True)
X = data["X"].astype(np.float32)
y = data["labels"].astype(np.int64)
clip_ids = data["clip_ids"]
label_names = list(data["label_names"])
species = data["species"]
is_train = data["is_train"]
print(f"X: {X.shape}, classes: {len(label_names)}")
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {label_names[u]}: {c}")

scaler = StandardScaler()
Xz = scaler.fit_transform(X).astype(np.float32)


## Helper: stratified label mask


In [ ]:
def stratified_mask(y, frac, rng):
    mask = np.zeros(y.shape[0], dtype=bool)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        n_lbl = max(1, int(round(frac * idx.size)))
        chosen = rng.choice(idx, size=n_lbl, replace=False)
        mask[chosen] = True
    return mask

LABEL_FRAC = 0.5
N_TRIALS = 20


## Reference run at default params (k=10, σ=median)

This is the baseline number to compare the parameter sweep against.


In [ ]:
W_default = build_knn_graph(Xz, k=10, sigma="median")

mu_for_lr = 1.0
if WINNER == "lap_reg":
    rng_cv = np.random.default_rng(123)
    mask_cv = stratified_mask(y, LABEL_FRAC, rng_cv)
    labeled_idx = np.where(mask_cv)[0]
    val_idx = rng_cv.choice(labeled_idx, size=max(4, int(0.2 * labeled_idx.size)), replace=False)
    train_mask = mask_cv.copy()
    train_mask[val_idx] = False
    best_acc = -1
    for mu in [0.01, 0.1, 1.0, 10.0, 100.0]:
        pred = laplacian_regularization(W_default, y, train_mask, mu=mu)
        acc = accuracy_score(y[val_idx], pred[val_idx])
        print(f"  mu={mu:>6.2f}  val acc={acc:.4f}")
        if acc > best_acc:
            best_acc = acc
            mu_for_lr = mu
    print(f"selected mu={mu_for_lr}")

def run_trial(W, mask):
    if WINNER == "lap_reg":
        pred = laplacian_regularization(W, y, mask, mu=mu_for_lr)
    else:
        pred = label_propagation(W, y, mask)
    unl = ~mask
    acc = accuracy_score(y[unl], pred[unl])
    f1 = f1_score(y[unl], pred[unl], average="macro",
                  labels=range(len(label_names)), zero_division=0)
    return acc, f1, pred

trial_masks = [stratified_mask(y, LABEL_FRAC, np.random.default_rng(2000 + t)) for t in range(N_TRIALS)]

t0 = time.time()
default_accs, default_f1s, default_preds = [], [], []
for mask in trial_masks:
    a, f, p = run_trial(W_default, mask)
    default_accs.append(a); default_f1s.append(f); default_preds.append(p)
print(f"default ({WINNER}, k=10, sigma=median): "
      f"acc {np.mean(default_accs):.4f} ± {np.std(default_accs):.4f}, "
      f"macro F1 {np.mean(default_f1s):.4f} ± {np.std(default_f1s):.4f}  "
      f"({time.time()-t0:.1f}s)")


## Parameter sensitivity sweep

Vary k and σ. For each (k, σ) we run all 20 trials and record mean accuracy.
This is the "mathematical meat" — it shows how graph construction affects
downstream classification.


In [ ]:
K_LIST = [3, 5, 7, 10, 15, 20]
SIGMA_LIST = ["0.5xmed", "1.0xmed", "2.0xmed", "5.0xmed", "self_tuning"]

def sigma_for(k, label, Xz):
    if label == "self_tuning":
        return "self_tuning"
    nn = NearestNeighbors(n_neighbors=k + 1).fit(Xz)
    d, _ = nn.kneighbors(Xz)
    med = float(np.median(d[:, 1:]))
    factor = float(label.split("xmed")[0])
    return factor * med

acc_grid = np.zeros((len(SIGMA_LIST), len(K_LIST)))
f1_grid = np.zeros_like(acc_grid)
gap_grid = np.zeros_like(acc_grid)

t0 = time.time()
for i, slabel in enumerate(SIGMA_LIST):
    for j, k in enumerate(K_LIST):
        sig = sigma_for(k, slabel, Xz)
        W = build_knn_graph(Xz, k=k, sigma=sig)
        eigs = spectral_gap_from_W(W, n_eigenvalues=10)
        diffs = np.diff(eigs)
        gap_grid[i, j] = float(diffs.max() if diffs.size else 0.0)

        accs, f1s = [], []
        for mask in trial_masks:
            a, f, _ = run_trial(W, mask)
            accs.append(a); f1s.append(f)
        acc_grid[i, j] = float(np.mean(accs))
        f1_grid[i, j] = float(np.mean(f1s))
        print(f"  sigma={slabel:<12s} k={k:>2d}  acc={acc_grid[i, j]:.4f}  gap={gap_grid[i, j]:.4f}")

print(f"\nSweep done in {time.time() - t0:.1f}s")


## Plots: parameter analysis


In [ ]:
plot_parameter_heatmap(K_LIST, SIGMA_LIST, acc_grid, FIG_DIR / "param_heatmap_acc.png",
                       title="Level 2 — accuracy over (k, σ)")

# Accuracy vs k at sigma=1.0xmed
fig, ax = plt.subplots(figsize=(7, 4))
i_med = SIGMA_LIST.index("1.0xmed")
ax.plot(K_LIST, acc_grid[i_med], marker="o", label="acc")
ax.set_xlabel("k (neighbors)"); ax.set_ylabel("mean accuracy")
ax.set_title(r"Level 2 — accuracy vs k ($\sigma$ = median)")
ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "acc_vs_k.png", dpi=150); plt.close(fig)

# Accuracy vs sigma at k=10
j_k10 = K_LIST.index(10)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(len(SIGMA_LIST)), acc_grid[:, j_k10], marker="o")
ax.set_xticks(range(len(SIGMA_LIST))); ax.set_xticklabels(SIGMA_LIST, rotation=20)
ax.set_ylabel("mean accuracy"); ax.set_title("Level 2 — accuracy vs σ (k = 10)")
ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "acc_vs_sigma.png", dpi=150); plt.close(fig)

# Spectral gap vs k at sigma=1.0xmed
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_LIST, gap_grid[i_med], marker="o")
ax.set_xlabel("k (neighbors)"); ax.set_ylabel("largest gap in first 10 eigenvalues")
ax.set_title(r"Level 2 — spectral gap vs k ($\sigma$ = median)")
ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "gap_vs_k.png", dpi=150); plt.close(fig)

best_i, best_j = np.unravel_index(np.argmax(acc_grid), acc_grid.shape)
best_k = K_LIST[best_j]; best_s = SIGMA_LIST[best_i]
print(f"best params: k={best_k}, sigma={best_s}, acc={acc_grid[best_i, best_j]:.4f}")


## Spectral gap at best params


In [ ]:
sig_best = sigma_for(best_k, best_s, Xz)
W_best = build_knn_graph(Xz, k=best_k, sigma=sig_best)
eigs_best = spectral_gap_from_W(W_best, n_eigenvalues=20)
plot_spectral_gap(eigs_best, FIG_DIR / "spectral_gap_best.png",
                  title=f"Level 2 — spectral gap (best k={best_k}, σ={best_s})")
print("first 20 eigenvalues:", eigs_best)


## Diffusion map (primary) + t-SNE (comparison)


In [ ]:
diff_coords = diffusion_map_embedding(W_best, n_components=3, t=5)
print("diffusion coords:", diff_coords.shape)

tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=0)
tsne_coords = tsne.fit_transform(Xz)
print("t-SNE coords:", tsne_coords.shape)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
import matplotlib as mpl
cmap = mpl.colormaps.get_cmap("tab20").resampled(len(label_names))
for ax, coords, title in [(axes[0], diff_coords[:, :2], "Diffusion map (t=5, best k & σ)"),
                           (axes[1], tsne_coords, "t-SNE")]:
    for i, name in enumerate(label_names):
        m = y == i
        ax.scatter(coords[m, 0], coords[m, 1], s=8, alpha=0.6, color=cmap(i), label=name)
    ax.set_title(f"Level 2 — {title}")
    ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
    ax.legend(fontsize=6, markerscale=1.5, loc="best", ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / "embeddings.png", dpi=150)
plt.close(fig)
print("saved embeddings.png")


## Diffusion times comparison (t = 1, 5, 20)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, t in zip(axes, [1, 5, 20]):
    coords = diffusion_map_embedding(W_best, n_components=2, t=t)
    for i, name in enumerate(label_names):
        m = y == i
        ax.scatter(coords[m, 0], coords[m, 1], s=8, alpha=0.6, color=cmap(i), label=name)
    ax.set_title(f"diffusion map t={t}")
    ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
fig.tight_layout()
fig.savefig(FIG_DIR / "diffusion_times.png", dpi=150)
plt.close(fig)
print("saved diffusion_times.png")


## ResNet-18 baseline


In [ ]:
spec = np.load(DATA_DIR / "level2_spectrograms.npz", allow_pickle=True)
S_all = spec["S"].astype(np.float32)
print("S:", S_all.shape)

mask0 = trial_masks[0]
train_keep = is_train & mask0
test_keep = ~is_train

S_tr = S_all[train_keep]; y_tr = y[train_keep]
S_te = S_all[test_keep]; y_te = y[test_keep]
print(f"CNN train: {S_tr.shape[0]}, test: {S_te.shape[0]}")

cnn_out = train_resnet_baseline(
    S_tr, y_tr, S_te, y_te,
    num_classes=len(label_names),
    epochs=15, batch_size=32, lr=1e-3, seed=0, verbose=True,
)
cnn_acc = accuracy_score(cnn_out["y_true"], cnn_out["y_pred"])
cnn_f1 = f1_score(cnn_out["y_true"], cnn_out["y_pred"], average="macro",
                  labels=range(len(label_names)), zero_division=0)
print(f"\nResNet-18: acc={cnn_acc:.4f}, macro F1={cnn_f1:.4f}")


## Final comparison & confusion matrix at best params


In [ ]:
# Get a representative prediction at best params for confusion matrix
acc_best, f1_best, pred_best = run_trial(W_best, trial_masks[0])
unl0 = ~trial_masks[0]

print("=== Level 2 results ===")
print(f"  graph SSL ({WINNER}, default k=10, sigma=median): "
      f"acc {np.mean(default_accs):.4f} ± {np.std(default_accs):.4f}  "
      f"macro F1 {np.mean(default_f1s):.4f} ± {np.std(default_f1s):.4f}")
print(f"  graph SSL ({WINNER}, BEST k={best_k}, sigma={best_s}): "
      f"acc {acc_grid[best_i, best_j]:.4f}  macro F1 {f1_grid[best_i, best_j]:.4f}")
print(f"  ResNet-18 baseline: acc {cnn_acc:.4f}  macro F1 {cnn_f1:.4f}")

plot_confusion_matrix(
    y[unl0], pred_best[unl0], label_names,
    save_path=FIG_DIR / f"confusion_matrix_{WINNER}_best.png",
    title=f"Level 2 — {WINNER} at best params (trial 1, unlabeled nodes)",
)
plot_confusion_matrix(
    cnn_out["y_true"], cnn_out["y_pred"], label_names,
    save_path=FIG_DIR / "confusion_matrix_resnet.png",
    title="Level 2 — ResNet-18 (HF test split)",
)
print("\nFigures saved to", FIG_DIR)
